# MLDJ Clean Baseline Notebook
This notebook builds a clean, leakage-aware dataset and trains a baseline model.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor


## Load Data

In [ ]:
df = pd.read_csv('eda_activities_slim.csv')
df['start_date'] = pd.to_datetime(df['start_date'], utc=True)
df = df[df['type'] == 'Run']
df = df.sort_values('start_date')


## Target: Average Pace (min/mile)
This target is derived from distance and moving_time. We will drop leakage features later.

In [ ]:
df['distance_miles'] = df['distance'] * 0.000621371
df['pace_min_per_mile'] = (df['moving_time'] / 60) / df['distance_miles']


## Temporal Features (no leakage)

In [ ]:
df['hour'] = df['start_date'].dt.hour
df['day_of_week'] = df['start_date'].dt.dayofweek
df['month'] = df['start_date'].dt.month
df['days_since_last_run'] = df['start_date'].diff() / pd.Timedelta(days=1)


## Rolling Training Load (time-based windows, shifted to avoid leakage)

In [ ]:
df = df.set_index('start_date')
df['miles_7d'] = df['distance_miles'].rolling('7D').sum().shift(1)
df['miles_30d'] = df['distance_miles'].rolling('30D').sum().shift(1)
df['hr_30d'] = df['average_heartrate'].rolling('30D').mean().shift(1)
df = df.reset_index()


## Drop Leakage Columns and Non-Predictive IDs

In [ ]:
drop_cols = [
    'activity_id','name','external_id',
    'distance','moving_time','elapsed_time',
    'average_speed','max_speed',
    'start_lat','start_lng','end_lat','end_lng',
    'pace_mean_min_per_mile','pace_std_min_per_mile','pace_drift_min_per_mile',
    'start_date'
]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])


## Define Features/Target and Split (time-based)

In [ ]:
df = df.dropna(subset=['pace_min_per_mile'])
df = df.sort_values('start_date') if 'start_date' in df.columns else df
y = df['pace_min_per_mile']
X = df.drop(columns=['pace_min_per_mile'])
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]


## Baseline Model

In [ ]:
model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train.select_dtypes(include=[np.number]), y_train)
preds = model.predict(X_test.select_dtypes(include=[np.number]))
mae = mean_absolute_error(y_test, preds)
rmse = mean_squared_error(y_test, preds, squared=False)
print(f'MAE: {mae:.3f} min/mi')
print(f'RMSE: {rmse:.3f} min/mi')
